# Ingesta de compañías productoras

Lee la carpeta `production_company` (CSV) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/production_companies` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

## Ingestion del folder production_company

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee la carpeta `production_company` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
production_companies_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/production_company")
) 
display(production_companies_df)

In [0]:
production_companies_df_shema = StructType([
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])
production_companies_df = (
    spark.read
        .format('csv')
        .schema(production_companies_df_shema)
        .load(f"{bronze_folder_path}/production_company")
)
display(production_companies_df)
production_companies_df.printSchema()

### Transformación de columnas

Renombra `companyId` y `companyName` a snake_case y agrega `ingestion_date` y `env`.

In [0]:
production_companies_df = (
    add_ingestion_date(production_companies_df)
    .select(
        col("companyId").alias("company_id"),
        col("companyName").alias("company_name"),
        lit(v_env).alias("env")
    )
)
display(production_companies_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/production_companies` en modo `overwrite`.

In [0]:
(
    production_companies_df.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{silver_folder_path}/production_companies")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")